### Advanced RAG Techniques

1. Chunking R&D (Day 4) -> different splitter, chunk sizes.

2. Encoder R&D - Best encoder based on a test set. PDFs are binary, convert pdf to markdown then vecotrise.

3. Prompt Improvement - content, current dates, relevant context and history.

4. Doc Pre-Processing - Use an LLM to reformat so chunks/text are optimised for chunking and querying.

5. Query Rewriting/Preprocessing - Use an LLM to convert the users questions (particularly follow up questions that may be combine) to a RAG Query

6. Query Expansion - Use LLM to turn question into multiple RAG Queries.

7. Re-Ranking - Use LLM to sub-select from RAG Results and reorder from the original retrieval to  most-least relevant.

8. Hierarchical - Use LLM to summarise at multiple levels. RAG look up at summary level then lookup at lower level.

9. Graph RAG - retrieve content closely related to similar documents. Only good if lots of relationships and can be handled through metadata.

10. Agentic Rag - Use agents (with memory and tools) for retrieval.

### We will now build a few without LangChain

In [1]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


### 1. Semantic Chunking

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [14]:
load_dotenv(override=True)

MODEL = 'gpt-4.1-nano'

DB_NAME = 'preprocessed_db'
collection_name = 'docs'
embedding_model = 'text-embedding-3-large'
#embedding_model = 'Qwen/Qwen3-Embedding-4B'
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

In [35]:
# copy of LangChains Document class
class Result(BaseModel):
    page_content: str
    metadata: dict

In [36]:
# a class to represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk. Only a few words likely surfaced from the chunk itself")
    summary: str = Field(description="A few sentences summarising the content of the chunk to answer common questions")
    original_text: str = Field("The original text of the chunk from the provided document")

    def as_result(self, document):
        metadata = {'source': document['source'], 'type': document['type']}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text, metadata = metadata)
    
class Chunks(BaseModel):
    chunks: list[Chunk]

#### Load & Chunk

1. Fetch documents from KB

2. Call LLM to turn docs into chunks

3. Store chunks in chroma

In [37]:
# Step 1 - mimick langchain doc loader

def fetch_documents():
    """  Homemade version of Langchain doc loader"""
    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob('*.md'): #recursive glob to search files matching pattern
            with open(file, 'r', encoding='utf-8') as f:
                documents.append({'type': doc_type, 'source': file.as_posix(), 'text': f.read()}) # as_posix converts file path into a string
    
    print(f'Loaded {len(documents)} documents')
    return documents


documents = fetch_documents()

Loaded 76 documents


In [38]:
# step 2 - making the chunks

def make_prompt(document):
    how_many = (len(document['text']) // AVERAGE_CHUNK_SIZE) + 1
    return f""" 
You take a document and split it into overlapping chunks for a knowledge base.

The document is from a shared drive of a company called Insurellm.
The document is of type: {document['type']}
The document has been retrieved from: {document['source']}

A chatbot will use these chunks to answer questions about the company.

You should divide up the document as you see fit, being sure that the entire docuemnt is returned in chunks maintaining all its contents.
Nothing must be left out.
This document should probably be split into {how_many} chunks, but you can have more or less where appropriate. Maintaining the right balance of context and relevance.

There should be overlap between chunks; typically 20% or 50 words.

For each chunk, you should provide a headline, summary, and the original text of the chunk.

Together all chunks must represent the document in its entirety with overlap.

Here is the document: {document['text']}

Respond with the chunks.

"""

In [39]:
documents[0]

{'type': 'products',
 'source': 'knowledge-base/products/Rellm.md',
 'text': "# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.\n\n### Seamless Integ

In [40]:
make_prompt(documents[0])

" \nYou take a document and split it into overlapping chunks for a knowledge base.\n\nThe document is from a shared drive of a company called Insurellm.\nThe document is of type: products\nThe document has been retrieved from: knowledge-base/products/Rellm.md\n\nA chatbot will use these chunks to answer questions about the company.\n\nYou should divide up the document as you see fit, being sure that the entire docuemnt is returned in chunks maintaining all its contents.\nNothing must be left out.\nThis document should probably be split into 8 chunks, but you can have more or less where appropriate. Maintaining the right balance of context and relevance.\n\nThere should be overlap between chunks; typically 20% or 50 words.\n\nFor each chunk, you should provide a headline, summary, and the original text of the chunk.\n\nTogether all chunks must represent the document in its entirety with overlap.\n\nHere is the document: # Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Sol

In [41]:
def make_messages(document):
    return [{"role": 'user', 'content': make_prompt(document)},]

In [42]:
def process_document(document):
    messages = make_messages(document) # create messages for model
    response = completion(model=MODEL, messages=messages, response_format=Chunks) # homemade chat completion to call llm
    reply = response.choices[0].message.content #store response content - a Json string of chunks -> {'chunks':[ {'text': 'chunk1} , {'text':'chunk2'} ] }
    doc_as_chunks = Chunks.model_validate_json(reply).chunks #Pydantic model to parse response and validates it is in expected schema
    return [chunk.as_result(document) for chunk in doc_as_chunks] #use as_result function for each chunk in doc_as_chunks

In [ ]:
#returns list of result objects in form of our pydantic structure
process_document(documents[0])

[Result(page_content='Product Overview: Rellm\n\nRellm is an AI-powered enterprise reinsurance platform developed by Insurellm, aimed at transforming risk management, decision-making, and operational efficiency in reinsurance firms.\n\n# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.', metadata={'source': 'knowledge-base/products/Rellm.md', 'type': 'products'}),
 Result(page_content="Key Features of Rellm: Part 1\n\nRellm offers AI-drive

In [46]:
# function to iterate through our documents and extend into our chunks list
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents): #tqdm adds progress bar as it iterates
        chunks.extend(process_document(doc))
    return chunks

chunks = create_chunks(documents)


100%|██████████| 76/76 [07:54<00:00,  6.24s/it]


In [ ]:
print(len(chunks))

#### Create embeddings and store in Chroma

In [ ]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME) #creates chroma client using folcer path on disk
    if collection_name in [c.name for c in chroma.list_collections()]: #clears collection if it already exists
        chroma.delete_collection(collection_name)
    
    texts = [chunk.page_content for chunk in chunks] #extracts text from each chunk (page_content from our Result object)
    emb = openai.embeddings.create(model=embedding_model, input=texts).data #sends chunks to openai embedding api
    vectors = [e.embedding for e in emb] #pulls raw vectors from api response

    collection = chroma.get_or_create_collection(collection_name) #creates and loads the collection

    ids = [str(i) for i in range(len(chunks))] #gens unique IDs for each chunk
    metas = [chunk.metadata for chunk in chunks] #extracts metadata from each chunk (metadata in as_result() from process_docs())

    collection.add(ids=ids, embeddings = vectors, documents = texts, metadatas=metas) #stores everything in chroma vectorstore
    print(f"Vectorstore created with {collection.count()} documents")

In [48]:
create_embeddings(chunks)

Vectorstore created with 408 documents


In [53]:
# lets visualise

chroma = PersistentClient(path=DB_NAME)
collection =chroma.get_or_create_collection(collection_name)
result =collection.get(include =['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]


In [54]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

#2d scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text = [f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo = 'text'
)])

fig.update_layout(title = '2D Chroma Vector Store Visulaisation',
                  scene=dict(xaxis_title='x', yaxis_title='y'),
                  width=800,
                  height=600,
                  margin = dict(r=20, b=10, l=10, t=10)
                  )

fig.show()

## 2. Advanced RAG -ReRank & Query Rewrite

In [55]:
class RankOrder(BaseModel):
    order: list[int] = Field(description='The order of relevance of chunks, from most relevant to least, by chunk ID number')

In [56]:
def rerank(question, chunks):
    system_prompt = """
    You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt =f" The user has asked the following question: \n\n{question}\n\nOrder all the chunks by text by relevance to the question."
    user_prompt += 'here are the chunks:\n\n'
    for index, chunk in enumerate(chunks):
        user_prompt += f"# Chunk ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    
    user_prompt+= 'reply only with the list of chunk ids, nothing else'

    messages =[
        {"role":'system', 'content': system_prompt},
        {"role":'user', 'content': user_prompt}
    ]

    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    return [chunks[i-1] for i in order]


In [57]:
RETRIEVAL_K=10

In [58]:
def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results['documents'][0], results['metadatas'][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks


In [60]:
question = 'who won the IIOTY award?'
chunks = fetch_context_unranked(question)

for chunk in chunks:
    print(chunk.page_content[:15]+'...')

Priya Sharma's ...
Additional HR N...
Annual Performa...
Performance His...
Career Progress...
Performance Rat...
Career Progress...
Educational Bac...
Career Progress...
Performance and...


In [61]:
reranked = rerank(question, chunks)
reranked

[Result(page_content='Additional HR Notes and Initiatives Involving Maxine Thompson\n\nMaxine has been involved in various trainings, received awards, and participates in company initiatives such as women-in-tech and mentorship programs. Development areas include improving stakeholder communication.\n\n## Other HR Notes\n- Maxine participated in various company-sponsored trainings related to big data technologies and cloud infrastructure.  \n- She was recognized for her contributions with the prestigious Insurellm IIOTY Innovator Award in 2023.  \n- Maxine is currently involved in the women-in-tech initiative and participates in mentorship programs to guide junior employees.  \n- Future development areas include improving her stakeholder communication skills to ensure smoother project transitions and collaboration.', metadata={'type': 'employees', 'source': 'knowledge-base/employees/Maxine Thompson.md'}),
 Result(page_content="Performance History - 2015 to 2023\n\nAvery's annual perfor

In [62]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [63]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [64]:
# in context, include source of the chunk

def make_rag_messages(question, history, chunk):
    context = '\n\n'.join(f"Extracted from {chunk.metadata['source']}:\n{chunk.page_content} " for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": 'system', 'content': system_prompt}] + history + [{'role':'user', 'content': question}]

In [65]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [66]:
rewrite_query("Who won the IIOTY", [])

'Who was the winner of the IIOTY award?'

In [67]:
def answer_question(question:str, history: list[dict] = [])->tuple[str, list]:
    """  
    Answer a question using RAG and return the answer and retrieved context
    """

    query = rewrite_query(question)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages = messages)
    return response.choices[0].message.content

In [68]:
answer_question('Who won the IIOTY award', [])

Who is the winner of the IIOTY award?


'Maxine Thompson was recognized with the prestigious Insurellm IIOTY Innovator Award in 2023.'

**Rewriting can dilute RAG Capabilities by adding in words to the query**

Looking at Pro-implementation -> make sure imports are adjusted in eval.py

Ingest.py

* Uses tenacity to aid rate limiting by performing exponential backoff. 
* Uses multiprocessing (Pool) to create chunks instead of iterating it chunks 5 times faster. Workers variable determines how many chunks are created at a time.


Answer.py

* Uses open source model
* retrieves 20 but reranks for top 10
* Introduces fetch context to enable Query Expansion (Due to Query Rewriting diluting RAG Lookup through adding extra words) - use both queries for rag lookup. Uses merge chunks then we rerank vs original question. -> this can be done with rewriter returning multiple rewrites


GPT-OSS -> great for generating markdown (great for formatting)